# Model Training

## Objective

The objective of this notebook is to train multiple machine learning models for customer churn prediction using the feature engineered dataset.

Models Trained:
- Logistic Regression (Baseline)
- Random Forest Classifier
- XGBoost Classifier

The models are evaluated using 5-Fold Cross Validation. The best performing model will be selected for detailed evaluation in the next notebook.

In [20]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

In [21]:
from xgboost import XGBClassifier

print("XGBoost Installed Successfully")

XGBoost Installed Successfully


In [22]:
# =====================================================
# Load Feature Engineered Dataset
# =====================================================
df = pd.read_csv("../data/processed/feature_engineered_data.csv")

print(df.shape)

df.head()

(7043, 44)


,Age,Number of Dependents,Population,Number of Referrals,Tenure in Months,Avg Monthly Long Distance Charges,Avg Monthly GB Download,Monthly Charge,Total Charges,Total Refunds,...,Streaming TV_Yes,Streaming Movies_Yes,Streaming Music_Yes,Unlimited Data_Yes,Contract_One Year,Contract_Two Year,Paperless Billing_Yes,Payment Method_Credit Card,Payment Method_Mailed Check,Churn Label
0,1.880110,-0.486835,2.201392,-0.650409,-1.278988,-1.486303,-0.612975,-0.834611,-0.988823,-0.248313,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1
1,1.641292,0.551874,1.585200,-0.317185,-0.993743,1.676120,-0.172176,0.528063,-0.726848,-0.248313,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1
2,1.462179,2.629292,1.200630,-0.650409,-0.586250,-0.752828,1.542040,1.019955,-0.232929,5.523605,...,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1
3,1.880110,0.551874,0.266580,-0.317185,-0.301005,-0.207092,-0.417064,1.121324,0.103315,1.451245,...,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1
4,1.999519,0.551874,0.195046,-0.317185,0.187986,-1.076516,-0.319109,0.390134,0.259379,-0.248313,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1


In [23]:
# =====================================================
# Feature Matrix and Target Variable
# =====================================================

X = df.drop("Churn Label", axis=1)

y = df["Churn Label"]

print("Feature Matrix :", X.shape)

print("Target Variable :", y.shape)

Feature Matrix : (7043, 43)
Target Variable : (7043,)


In [24]:
# =====================================================
# Train Test Split
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Samples :", X_train.shape)

print("Testing Samples :", X_test.shape)

Training Samples : (5634, 43)
Testing Samples : (1409, 43)


# Logistic Regression

Logistic Regression is used as the baseline classification model for churn prediction.

In [25]:
# =====================================================
# Logistic Regression Pipeline
# =====================================================

logistic_pipeline = Pipeline([

    ("scaler", StandardScaler()),

    ("model", LogisticRegression(random_state=42))

])

logistic_pipeline.fit(X_train, y_train)

logistic_accuracy = logistic_pipeline.score(X_test, y_test)

print("Logistic Regression Accuracy")

print(round(logistic_accuracy,4))

Logistic Regression Accuracy
0.9638


# Random Forest

Random Forest is an ensemble learning algorithm capable of capturing complex decision boundaries.

In [26]:
# =====================================================
# Random Forest Pipeline
# =====================================================

rf_pipeline = Pipeline([

    ("scaler", StandardScaler()),

    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))

])

rf_pipeline.fit(X_train,y_train)

rf_accuracy = rf_pipeline.score(X_test,y_test)

print("Random Forest Accuracy")

print(round(rf_accuracy,4))

Random Forest Accuracy
0.9553


# XGBoost

XGBoost is a gradient boosting algorithm designed for high predictive performance.

In [27]:
# =====================================================
# XGBoost Pipeline
# =====================================================

xgb_pipeline = Pipeline([

    ("scaler", StandardScaler()),

    ("model", XGBClassifier(

        random_state=42,

        eval_metric="logloss"

    ))

])

xgb_pipeline.fit(X_train,y_train)

xgb_accuracy = xgb_pipeline.score(X_test,y_test)

print("XGBoost Accuracy")

print(round(xgb_accuracy,4))

XGBoost Accuracy
0.9574


# Cross Validation

5-Fold Cross Validation is performed to evaluate the stability and generalization capability of each model.

In [28]:
# =====================================================
# Cross Validation
# =====================================================

log_cv = cross_val_score(
    logistic_pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

rf_cv = cross_val_score(
    rf_pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

xgb_cv = cross_val_score(
    xgb_pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

In [29]:
# =====================================================
# Model Comparison
# =====================================================

comparison = pd.DataFrame({

    "Model":[

        "Logistic Regression",

        "Random Forest",

        "XGBoost"

    ],

    "Test Accuracy":[

        logistic_accuracy,

        rf_accuracy,

        xgb_accuracy

    ],

    "Cross Validation Accuracy":[

        log_cv.mean(),

        rf_cv.mean(),

        xgb_cv.mean()

    ]

})

comparison = comparison.sort_values(

    by="Cross Validation Accuracy",

    ascending=False

)

comparison.reset_index(drop=True,inplace=True)

comparison

,Model,Test Accuracy,Cross Validation Accuracy
0,XGBoost,0.957417,0.952011
1,Logistic Regression,0.963804,0.920783
2,Random Forest,0.955287,0.905312


In [30]:
# =====================================================
# Best Performing Model
# =====================================================

best_model = comparison.iloc[0]

print("Best Model")

print(best_model["Model"])

print()

print("Cross Validation Accuracy")

print(round(best_model["Cross Validation Accuracy"],4))

Best Model
XGBoost

Cross Validation Accuracy
0.952


## Observation

- Logistic Regression served as the baseline classifier.
- Random Forest improved predictive performance using ensemble learning.
- XGBoost achieved the highest overall performance based on cross-validation accuracy.
- The best model will be evaluated further using Confusion Matrix, ROC-AUC, Precision, Recall and F1-Score in the next notebook.